# Retrieval Evaluation — Generic\nEvaluate retrieval quality for **any** ChromaDB collection (Cohere or OpenRouter provider).\n\n**Workflow**\n1. Configure provider, model, collection names, and question datasets in **Cell 1**.\n2. Run all cells in order.\n3. Metrics (Recall@k, Accuracy@k, MRR@k) are printed at the end, both overall and broken down by question type and source pair.

In [17]:
os.getcwd()

'<PROJECT_ROOT>'

In [1]:
# ============================================================
# Cell 1 — Configuration
# Edit the variables below, then run all cells.
# ============================================================
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(os.getcwd(), "..", ".env"))

from scripts.config import (
    DATA_DIR,
    EMBEDDINGS_1887_COHERE,
    EMBEDDINGS_1887_OPENROUTER,
    COHERE_API_KEY,
    OPENROUTER_API_KEY,
    COHERE_EMBEDDING_MODEL,
    OPENROUTER_EMBEDDING_MODEL,
)


import chromadb
from chromadb.utils import embedding_functions
from scripts.utils.embedding_functions import OpenRouterEmbeddingFunction



# ── Provider ──────────────────────────────────────────────────
PROVIDER = "cohere"           # "cohere" or "openrouter"
CHROMA_DIR = EMBEDDINGS_1887_COHERE if PROVIDER == "cohere" else EMBEDDINGS_1887_OPENROUTER
EMBEDDING_MODEL = "google/gemini-embedding-2-preview"
EMBEDDING_MODEL = COHERE_EMBEDDING_MODEL if PROVIDER == "cohere" else OPENROUTER_EMBEDDING_MODEL

client = chromadb.PersistentClient(path=CHROMA_DIR)


# ── Collections to search ────────────────────────────────────
# Each entry: {"name": "<chroma collection name>", "label": "<human label>"}
COLLECTION_CONFIGS = [
    {"name": "debattre_1887_10000_hierarchical_cohere", "label": "Les Débats (Cohere)"},
    {"name": "lintransigeant_1887_v1",              "label": "L'Intransigeant (Cohere)"},
    {"name": "legaulois_1887_v1",                   "label": "Le Gaulois (Cohere)"},
]



# ── Question datasets ─────────────────────────────────────────
QUESTIONS_DIR = os.path.join(DATA_DIR, "questions")
QUESTION_DATASETS = [
    {
        "name": "Generic_LeGaulois_Debats",
        "file": "MH_Generic_legaulois_v1.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "Les Débats",
        "question_type": "MH - Generic",
    },
    {
        "name": "Generic_LIntransigeant_Debats",
        "file": "generated_questions_lintransigeant_all.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "L'Intransigeant",
        "source2_name": "Les Débats",
        "question_type": "MH - Generic",
    },
    {
        "name": "BridgeEntity_LIntransigeant_Debats",
        "file": "MH_BridgeEntity_lintransigeant_v2.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "L'Intransigeant",
        "source2_name": "Les Débats",
        "question_type": "MH - BridgeEntity",
    },
    {
        "name": "Comparative_LeGaulois_LIntransigeant",
        "file": "MH_Comparative_legaulois_lintransigeant_v3.jsonl",
        "column_mapping": {
            "id_1": "legaulois_id", "id_2": "lintransigeant_id",
            "doc_1": "legaulois_doc", "doc_2": "lintransigeant_doc",
            "metadata_1": "legaulois_metadata", "metadata_2": "lintransigeant_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "L'Intransigeant",
        "question_type": "MH - Comparative",
    },
    {
        "name": "Generic_LeGaulois_LIntransigeant",
        "file": "MH_Generic_legaulois_lintransigeant_v3.jsonl",
        "column_mapping": {
            "id_1": "legaulois_id", "id_2": "lintransigeant_id",
            "doc_1": "legaulois_doc", "doc_2": "lintransigeant_doc",
            "metadata_1": "legaulois_metadata", "metadata_2": "lintransigeant_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "L'Intransigeant",
        "question_type": "MH - Generic",
    },
]

# ── Retrieval & evaluation settings ──────────────────────────
N_RESULTS = 10
K_VALUES  = [1, 2, 3, 5, 10]

# ── Output ────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(DATA_DIR, "retrieval")
# File name is derived automatically from provider + model
_model_slug = EMBEDDING_MODEL.replace("/", "_").replace("-", "_")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"retrieval_results_{PROVIDER}_{_model_slug}.jsonl")


print(f"Provider     : {PROVIDER}")
print(f"Model        : {EMBEDDING_MODEL}")
print(f"Chroma dir   : {CHROMA_DIR}")
print(f"Collections  : {[c['label'] for c in COLLECTION_CONFIGS]}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"N_RESULTS    : {N_RESULTS}  |  K_VALUES: {K_VALUES}")

Provider     : cohere
Model        : embed-v4.0
Chroma dir   : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\embeddings_1887_cohere
Collections  : ['Les Débats (Cohere)', "L'Intransigeant (Cohere)", 'Le Gaulois (Cohere)']
Output file  : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\retrieval\retrieval_results_cohere_embed_v4.0.jsonl
N_RESULTS    : 10  |  K_VALUES: [1, 2, 3, 5, 10]


In [13]:
if PROVIDER == "cohere":
    ef = embedding_functions.CohereEmbeddingFunction(
        api_key=COHERE_API_KEY, model_name=EMBEDDING_MODEL
    )
else:
    ef = OpenRouterEmbeddingFunction(api_key=OPENROUTER_API_KEY, model=EMBEDDING_MODEL)

collections = []
for cfg in COLLECTION_CONFIGS:
    try:
        col = client.get_collection(name=cfg["name"], embedding_function=ef)
        collections.append(col)
        print(f"  ✓  {cfg['label']}  ({col.count()} docs)")
    except Exception as e:
        print(f"  ✗  {cfg['label']}  — {e}")

print(f"\nLoaded {len(collections)} / {len(COLLECTION_CONFIGS)} collections.")

  ✓  Les Débats (Cohere)  (5103 docs)
  ✓  L'Intransigeant (Cohere)  (79 docs)
  ✓  Le Gaulois (Cohere)  (78 docs)

Loaded 3 / 3 collections.


In [14]:
# ============================================================
# Cell 3 — Load question datasets
# ============================================================
import pandas as pd
from scripts.utils.questions import create_questions_dataset

all_questions = []

for ds in QUESTION_DATASETS:
    path = os.path.join(QUESTIONS_DIR, ds["file"])
    if not os.path.exists(path):
        print(f"  WARNING — file not found: {path}")
        continue
    df_q = create_questions_dataset(
        jsonl_path=path,
        column_mapping=ds["column_mapping"],
        source1_name=ds["source1_name"],
        source2_name=ds["source2_name"],
        question_type=ds["question_type"],
    )
    all_questions.append(df_q)
    print(f"  {ds['name']}: {len(df_q)} questions")

df = pd.concat(all_questions, ignore_index=True)
print(f"\nTotal questions : {len(df)}")
print(f"Question types  : {df['question_type'].value_counts().to_dict()}")

Created dataset with 220 questions from 77 original entries
  Generic_LeGaulois_Debats: 220 questions
Created dataset with 199 questions from 79 original entries
  Generic_LIntransigeant_Debats: 199 questions
Created dataset with 152 questions from 79 original entries
  BridgeEntity_LIntransigeant_Debats: 152 questions
Created dataset with 192 questions from 47 original entries
  Comparative_LeGaulois_LIntransigeant: 192 questions
Created dataset with 122 questions from 48 original entries
  Generic_LeGaulois_LIntransigeant: 122 questions

Total questions : 885
Question types  : {'MH - Generic': 541, 'MH - Comparative': 192, 'MH - BridgeEntity': 152}


In [15]:
# ============================================================
# Cell 4 — Perform retrieval
# Results are appended incrementally to OUTPUT_FILE so the cell
# can be safely re-run after an interruption.
# ============================================================
from scripts.retrieval_utils import RAGRetriever, perform_retrieval

retriever = RAGRetriever(collections, n_results=N_RESULTS)
perform_retrieval(df, retriever, output_file=OUTPUT_FILE)
print(f"\nResults written to: {OUTPUT_FILE}")

Found 0 existing retrieval results
Retrieving for 885 questions...


  0%|          | 0/885 [00:00<?, ?it/s]

100%|██████████| 885/885 [09:27<00:00,  1.56it/s]


Retrieval complete! Results saved to c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\retrieval_results_cohere_embed_v4.0.jsonl
Results also saved as CSV → c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\retrieval_results_cohere_embed_v4.0.csv

Results written to: c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\retrieval_results_cohere_embed_v4.0.jsonl


In [16]:
# ============================================================
# Cell 5 — Evaluate all questions
# ============================================================
from scripts.retrieval_utils import evaluate_and_print_metrics
from scripts.utils.json_reading import read_jsonl_basic

df_result = pd.DataFrame(read_jsonl_basic(OUTPUT_FILE))
print(f"Loaded {len(df_result)} retrieval results from {OUTPUT_FILE}")

result_all = evaluate_and_print_metrics(
    df_result,
    dataset_name=f"ALL  ({PROVIDER} / {EMBEDDING_MODEL})",
    k_values=K_VALUES,
)

Loaded 885 retrieval results from c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\retrieval_results_cohere_embed_v4.0.jsonl

EVALUATION RESULTS FOR: ALL  (COHERE / EMBED-V4.0)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 15931.09it/s]



Recall Metrics for 885 questions:
  recall@1: 0.182
  recall@2: 0.295
  recall@3: 0.360
  recall@5: 0.442
  recall@10: 0.545


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 15735.84it/s]



Accuracy Metrics for 885 questions:
  accuracy@1: 0.000
  accuracy@2: 0.096
  accuracy@3: 0.155
  accuracy@5: 0.250
  accuracy@10: 0.373


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 9893.04it/s]


MRR Metrics for 885 questions:
  mrr@1: 0.182
  mrr@2: 0.239
  mrr@3: 0.261
  mrr@5: 0.279
  mrr@10: 0.293


In [6]:
# ============================================================
# Cell 5 — Evaluate all questions
# ============================================================
from scripts.retrieval_utils import evaluate_and_print_metrics
from scripts.utils.json_reading import read_jsonl_basic

df_result = pd.DataFrame(read_jsonl_basic(OUTPUT_FILE))
print(f"Loaded {len(df_result)} retrieval results from {OUTPUT_FILE}")

result_all = evaluate_and_print_metrics(
    df_result,
    dataset_name=f"ALL  ({PROVIDER} / {EMBEDDING_MODEL})",
    k_values=K_VALUES,
)

Loaded 885 retrieval results from c:\Users\user\Desktop\workdir\ORDER_EMNLP2026\data\retrieval\retrieval_results_openrouter_google_gemini_embedding_2_preview.jsonl

EVALUATION RESULTS FOR: ALL  (OPENROUTER / GOOGLE/GEMINI-EMBEDDING-2-PREVIEW)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 7230.32it/s]



Recall Metrics for 885 questions:
  recall@1: 0.228
  recall@2: 0.304
  recall@3: 0.330
  recall@5: 0.372
  recall@10: 0.409


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 6820.90it/s]



Accuracy Metrics for 885 questions:
  accuracy@1: 0.006
  accuracy@2: 0.038
  accuracy@3: 0.045
  accuracy@5: 0.069
  accuracy@10: 0.082


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 2844.43it/s]


MRR Metrics for 885 questions:
  mrr@1: 0.228
  mrr@2: 0.266
  mrr@3: 0.274
  mrr@5: 0.284
  mrr@10: 0.289


In [7]:
# ============================================================
# Cell 6 — Breakdown by question type
# ============================================================
for qtype in sorted(df_result["question_type"].unique()):
    subset = df_result[df_result["question_type"] == qtype]
    evaluate_and_print_metrics(subset, dataset_name=qtype, k_values=K_VALUES)


EVALUATION RESULTS FOR: MH - BRIDGEENTITY

Computing recall metrics...


100%|██████████| 152/152 [00:00<00:00, 16996.38it/s]



Recall Metrics for 152 questions:
  recall@1: 0.253
  recall@2: 0.319
  recall@3: 0.342
  recall@5: 0.372
  recall@10: 0.391


Computing Accuracy: 100%|██████████| 152/152 [00:00<00:00, 5982.75it/s]



Accuracy Metrics for 152 questions:
  accuracy@1: 0.000
  accuracy@2: 0.007
  accuracy@3: 0.013
  accuracy@5: 0.013
  accuracy@10: 0.013


...


Recall Metrics for 541 questions:
  recall@1: 0.229
  recall@2: 0.309
  recall@3: 0.334
  recall@5: 0.370
  recall@10: 0.401


Computing Accuracy: 100%|██████████| 541/541 [00:00<00:00, 10411.57it/s]



Accuracy Metrics for 541 questions:
  accuracy@1: 0.004
  accuracy@2: 0.030
  accuracy@3: 0.035
  accuracy@5: 0.054
  accuracy@10: 0.068


Computing MRR: 100%|██████████| 541/541 [00:00<00:00, 4551.83it/s]


MRR Metrics for 541 questions:
  mrr@1: 0.229
  mrr@2: 0.269
  mrr@3: 0.277
  mrr@5: 0.285
  mrr@10: 0.289


In [8]:
# ============================================================
# Cell 7 — Breakdown by source pair
# ============================================================
df_result["source_pair"] = df_result["source_1"] + "  ×  " + df_result["source_2"]
for pair in sorted(df_result["source_pair"].unique()):
    subset = df_result[df_result["source_pair"] == pair]
    evaluate_and_print_metrics(subset, dataset_name=pair, k_values=K_VALUES)


EVALUATION RESULTS FOR: L'INTRANSIGEANT  ×  LES DÉBATS

Computing recall metrics...


100%|██████████| 351/351 [00:00<00:00, 18757.73it/s]



Recall Metrics for 351 questions:
  recall@1: 0.264
  recall@2: 0.340
  recall@3: 0.372
  recall@5: 0.392
  recall@10: 0.415


Computing Accuracy: 100%|██████████| 351/351 [00:00<00:00, 19217.57it/s]



Accuracy Metrics for 351 questions:
  accuracy@1: 0.000
  accuracy@2: 0.009
  accuracy@3: 0.011
  accuracy@5: 0.011
  accuracy@10: 0.011


...


Recall Metrics for 220 questions:
  recall@1: 0.170
  recall@2: 0.223
  recall@3: 0.232
  recall@5: 0.250
  recall@10: 0.270


Computing Accuracy: 100%|██████████| 220/220 [00:00<00:00, 18337.21it/s]



Accuracy Metrics for 220 questions:
  accuracy@1: 0.000
  accuracy@2: 0.000
  accuracy@3: 0.000
  accuracy@5: 0.000
  accuracy@10: 0.000


Computing MRR: 100%|██████████| 220/220 [00:00<00:00, 10512.28it/s]


MRR Metrics for 220 questions:
  mrr@1: 0.170
  mrr@2: 0.197
  mrr@3: 0.200
  mrr@5: 0.203
  mrr@10: 0.206


In [ ]:
# ============================================================
# Cell 8 — Summary table (Recall@k / Accuracy@k / MRR@k per group)
# ============================================================
import numpy as np
from scripts.retrieval_utils import (
    compute_recall_metrics_dataframe,
    compute_accuracy_at_k,
    compute_mrr_at_k,
)

def _metrics_row(subset, k_values):
    recall   = compute_recall_metrics_dataframe(subset, k_values=k_values)
    accuracy = compute_accuracy_at_k(subset, k_values=k_values)
    mrr      = compute_mrr_at_k(subset, k_values=k_values)
    row = {"n": len(subset)}
    for k in k_values:
        row[f"R@{k}"]   = round(np.mean(recall[f"recall@{k}"]),   3)
        row[f"Acc@{k}"] = round(np.mean(accuracy[f"accuracy@{k}"]), 3)
        row[f"MRR@{k}"] = round(np.mean(mrr[f"mrr@{k}"]),         3)
    return row

groups = {"ALL": df_result}
for qtype in sorted(df_result["question_type"].unique()):
    groups[qtype] = df_result[df_result["question_type"] == qtype]
for pair in sorted(df_result["source_pair"].unique()):
    groups[pair]  = df_result[df_result["source_pair"] == pair]

rows = []
for label, subset in groups.items():
    r = _metrics_row(subset, K_VALUES)
    r["group"] = label
    rows.append(r)

df_summary = pd.DataFrame(rows).set_index("group")
print(f"\nSummary — {PROVIDER} / {EMBEDDING_MODEL}")
display(df_summary)